<a href="https://colab.research.google.com/github/MayerT1/Prep_GEDI/blob/main/prep_inference_75Overlap__matching_GroundSampleArea.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install rasterio shapely tqdm

import os
import shutil
import math
import numpy as np
from tqdm import tqdm
from zipfile import ZipFile
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.windows import from_bounds, transform as window_transform
from rasterio.transform import Affine
from shapely.geometry import box
import traceback
import re

# =========================================================
# 0. MOUNT DRIVE
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# 1. COPY INPUT RASTERS TO LOCAL COLAB STORAGE
# =========================================================
# Much faster and avoids Drive I/O bottlenecks
gdrive_in = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inference"
local_in = "/content/data_in"
local_out = "/content/chips"
zip_out_local = "/content/chips_preprocessed_Matching_ground.zip"
zip_out_gdrive = f"{gdrive_in}/chips_preprocessed_Matching_ground..zip"

os.makedirs(local_in, exist_ok=True)
os.makedirs(local_out, exist_ok=True)

input_names = [
    "s1Ascending_2024.tif",
    "DEMindices_2024.tif",
    "HLS_2024.tif",
    "LandsatComposite_2024.tif",
    "LandsatIndices_2024.tif",
    "landsatTasseledCapIndices_2024.tif",
    "S2Composite_2024.tif",
    "S2Indices_2024.tif",
]

# Copy from Drive → local
for fn in input_names:
    shutil.copy(f"{gdrive_in}/{fn}", f"{local_in}/{fn}")
print("All input rasters copied to fast local storage.")

REFERENCE_RASTER = f"{local_in}/s1Ascending_2024.tif"

# =========================================================
# 2. CONFIG
# =========================================================
TILE_METERS = 320.0
OVERLAP_FRACTION = 0.75

# =========================================================
# 3. FUNCTIONS
# =========================================================
def parse_name_and_year(path):
    fname = os.path.basename(path)
    stem = re.sub(r'\.tif+f?$', '', fname, flags=re.IGNORECASE)
    m = re.search(r'(19|20)\d{2}', stem)
    year = m.group(0) if m else "unknown"
    eo = re.sub(r'(_?(19|20)\d{2})', '', stem)
    eo = eo.strip('_-')
    if not eo:
        eo = stem
    return eo, year

def fill_nan_with_median(arr):
    if np.isnan(arr).all():
        return np.zeros_like(arr, np.float32)
    med = np.nanmedian(arr)
    return np.where(np.isnan(arr), med, arr).astype(np.float32)

def minmax_norm(arr):
    mn = arr.min()
    mx = arr.max()
    if mx - mn == 0:
        return np.zeros_like(arr, dtype=np.float32)
    return ((arr - mn) / (mx - mn)).astype(np.float32)

def bounds_for_ref_window(x_off, y_off, tile_px, transform):
    left = transform.c + x_off * transform.a
    top = transform.f + y_off * transform.e
    right = left + tile_px * transform.a
    bottom = top + tile_px * transform.e
    return min(left,right), min(top,bottom), max(left,right), max(top,bottom)

# =========================================================
# 4. BUILD GRID FROM REF RASTER
# =========================================================
with rasterio.open(REFERENCE_RASTER) as ref:
    T = ref.transform
    px = abs(T.a)
    tile_px = int(round(TILE_METERS / px))
    stride_px = max(1, int(round(tile_px * (1 - OVERLAP_FRACTION))))

    print("Pixel size:", px)
    print("Tile (px):", tile_px)
    print("Stride (px):", stride_px)

    xs = list(range(0, max(1, ref.width - tile_px + 1), stride_px))
    ys = list(range(0, max(1, ref.height - tile_px + 1), stride_px))
    if xs[-1] != ref.width - tile_px:
        xs.append(ref.width - tile_px)
    if ys[-1] != ref.height - tile_px:
        ys.append(ref.height - tile_px)

    NROWS, NCOLS = len(ys), len(xs)
    print(f"Grid: {NROWS} x {NCOLS}")

# =========================================================
# 5. PROCESS ALL RASTERS BUT WRITE ONLY TO /content/chips
# =========================================================
expected_outputs = []
written = []
errors = []

for r_idx, y0 in enumerate(tqdm(ys, desc="Rows")):
    for c_idx, x0 in enumerate(xs):
        minx, miny, maxx, maxy = bounds_for_ref_window(x0, y0, tile_px, T)

        for fn in input_names:
            ipath = f"{local_in}/{fn}"
            eo, year = parse_name_and_year(fn)
            outname = f"{eo}_{year}_{r_idx:04d}_{c_idx:04d}.tif"
            outpath = f"{local_out}/{outname}"
            expected_outputs.append(outname)

            try:
                with rasterio.open(ipath) as src:
                    dst = np.full((src.count, tile_px, tile_px), np.nan, dtype=np.float32)

                    dst_w = (maxx - minx) / tile_px
                    dst_h = (maxy - miny) / tile_px
                    dst_transform = Affine(dst_w, 0, minx, 0, -dst_h, maxy)

                    try:
                        s_win = from_bounds(minx, miny, maxx, maxy,
                                            src.transform,
                                            height=src.height, width=src.width)
                        s_win = s_win.round_offsets().round_shape()
                        s_data = src.read(window=s_win, boundless=True, fill_value=src.nodata)
                        s_T = window_transform(s_win, src.transform)
                    except Exception:
                        s_data = src.read()
                        s_T = src.transform

                    for b in range(src.count):
                        band_dst = np.empty((tile_px, tile_px), np.float32)
                        reproject(
                            source=s_data[b],
                            destination=band_dst,
                            src_transform=s_T,
                            src_crs=src.crs,
                            dst_transform=dst_transform,
                            dst_crs=ref.crs,
                            resampling=Resampling.bilinear
                        )
                        if src.nodata is not None:
                            band_dst = np.where(band_dst == src.nodata, np.nan, band_dst)

                        # fill NaN & normalize
                        band_dst = fill_nan_with_median(band_dst)
                        band_dst = minmax_norm(band_dst)
                        dst[b] = band_dst

                    profile = src.profile.copy()
                    profile.update({
                        "transform": dst_transform,
                        "crs": ref.crs,
                        "height": tile_px,
                        "width": tile_px,
                        "dtype": "float32",
                        "driver": "GTiff",
                        "compress": "lzw",
                        "count": src.count,
                    })

                    with rasterio.open(outpath, "w", **profile) as dstf:
                        dstf.write(dst)

                    written.append(outname)

            except Exception as e:
                errors.append((fn, r_idx, c_idx, str(e)))
                print("ERROR:", fn, r_idx, c_idx)
                traceback.print_exc()

# =========================================================
# 6. INVENTORY CHECK
# =========================================================
missing = sorted(list(set(expected_outputs) - set(written)))
print("Missing:", len(missing))
if missing:
    print(missing[:30])

# =========================================================
# 7. ZIP EVERYTHING (LOCAL ONLY)
# =========================================================
print("Creating ZIP...")
with ZipFile(zip_out_local, 'w') as zf:
    for f in os.listdir(local_out):
        if f.endswith(".tif"):
            zf.write(f"{local_out}/{f}", f)

# =========================================================
# 8. COPY ZIP TO DRIVE ONLY ONCE
# =========================================================
shutil.copy(zip_out_local, zip_out_gdrive)
print("ZIP copied to drive:", zip_out_gdrive)







Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All input rasters copied to fast local storage.
Pixel size: 10.0
Tile (px): 32
Stride (px): 8
Grid: 154 x 156


Rows: 100%|██████████| 154/154 [1:52:35<00:00, 43.87s/it]


Missing: 0
Creating ZIP...
ZIP copied to drive: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inference/chips_preprocessed_Matching_ground..zip


In [4]:
import os

folder = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inference/matching_GroundSampleArea_chipped_preprocessed"

prefixes = [
    "s1Ascending",
    "S2Composite",
    "S2Indices",
    "LandsatComposite",
    "LandsatIndices",
    "HLS",
    "landsatTasseledCapIndices",
    "DEMindices"
]

# Initialize counts
counts = {p: 0 for p in prefixes}

# Loop through directory
for fname in os.listdir(folder):
    for p in prefixes:
        if fname.startswith(p):
            counts[p] += 1
            break  # avoid double-counting if two prefixes overlap

# Print results
for p, c in counts.items():
    print(f"{p}: {c}")


s1Ascending: 2797
S2Composite: 2798
S2Indices: 2797
LandsatComposite: 2798
LandsatIndices: 2798
HLS: 2798
landsatTasseledCapIndices: 2797
DEMindices: 2798


confirm its done

In [6]:
!pip install descartes

In [7]:
import os
import rasterio
from rasterio.plot import show
from rasterio.windows import Window
import matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.ops import unary_union
from descartes import PolygonPatch

# Folder containing the chips
folder = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inference/matching_GroundSampleArea_chipped_preprocessed"

prefix = "s1Ascending"

footprints = []

# Collect footprints
for fname in os.listdir(folder):
    if fname.startswith(prefix) and fname.endswith(".tif"):
        path = os.path.join(folder, fname)
        with rasterio.open(path) as src:
            bounds = src.bounds
            # Create shapely polygon for the tile footprint
            footprints.append(box(bounds.left, bounds.bottom, bounds.right, bounds.top))

print(f"Collected {len(footprints)} footprints.")

# Plot
fig, ax = plt.subplots(figsize=(12, 12))
ax.set_title("Footprints of all s1Ascending chips")

for fp in footprints:
    patch = PolygonPatch(fp, alpha=0.3, linewidth=0.2)
    ax.add_patch(patch)

# Fit axes to all chips
union = unary_union(footprints)
minx, miny, maxx, maxy = union.bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

plt.show()


KeyboardInterrupt: 